In [ ]:
import os
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    spells_from_cross,
    spells_from_cross_catd_simple,
    extract_features,
    gaussian_smooth_func,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import create_bootstrapped_times, odds_ratio, is_signif_OR, common_OR

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/exp9")

ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets)
props = pl.read_parquet(basepath.joinpath("props.parquet"))
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
pers = slowness_from_cross(cross).drop("is_polar")
props = props.join(pers, on=["time", "jet ID"])

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col(
    "s"
).filter(over_europe).sum()
lat_over_europe = jets.group_by("time", "jet ID").agg(
    lat_over_europe.fill_nan(0).alias("lat_over_europe")
)
props = props.join(lat_over_europe, on=["time", "jet ID"])

props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
pers = props_catd.rolling("time", period="5d", group_by="jet").agg(pers=pl.col("slowness").sum())
props_catd = props_catd.join(pers, on=("time", "jet"))

cross_catd_ofile = basepath.joinpath("cross_catd.parquet")
if cross_catd_ofile.is_file():
    cross_catd = pl.read_parquet(cross_catd_ofile)
else:
    cross_catd = track_jets(phat_jets)
    cross_catd.write_parquet(cross_catd_ofile)
slowness_catd = slowness_from_cross(cross_catd).rename({"slowness": "slowness_catd"})
pers_catd = slowness_catd.rolling("time", period="5d", group_by="jet").agg(pers_catd=pl.col("slowness_catd").sum())
pers_catd = slowness_catd.join(pers_catd, on=("time", "jet"))
phat_filter = (pl.col("is_polar") < 0.5) | (
    (pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8)
)
phat_props = props.filter(phat_filter)
phat_props = squarify(average_jet_categories(phat_props), ["time", "jet"])
phat_props = phat_props.join(
    phat_props.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
phat_props = phat_props.join(pers_catd.drop("is_polar"), on=("time", "jet"))
phat_props_summer = summer_filter.join(phat_props, on="time")
props_catd_summer = summer_filter.join(props_catd, on="time")


spells_list = spells_from_cross(
    jets,
    cross,
    2.2e6,
    0.4,
    0.12,
    n_STJ=64,
    n_EDJ=30,
    season=summer,
)
filtering_for_STJ = props_catd.select("time", "jet", "mean_lat", "mean_s").pivot(on="jet", index="time").drop(cs.contains("EDJ"))
filtering_for_STJ = spells_list["STJ"].join(filtering_for_STJ, on="time").group_by("spell").agg(pl.col("mean_lat_STJ").mean())
filtering_for_STJ = filtering_for_STJ.filter(pl.col("mean_lat_STJ") > 33) 
spells_list["STJ"] = filtering_for_STJ.select("spell").join(spells_list["STJ"], on="spell")
spells_list["STJ"] = spells_list["STJ"].with_columns(pl.col("spell").rle_id()).sort("spell", "time")
spells_list["EDJ"] = spells_list["EDJ"].sort("spell", "time")
for jet, spells in spells_list.items():
    print(jet, spells["spell"].n_unique())
    print(jet, spells["len"].min() / 4)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

In [ ]:
from jetutils.jet_finding import connected_from_cross, slowness_expr
overlap_thresh = 0.7
dis_polar_thresh = 0.12
season = summer
n_spells = []
for dist_thresh in tqdm(np.linspace(.7e6, 4e6, 61)):
    _, summary_comp = connected_from_cross(
        jets,
        cross,
        dist_thresh=dist_thresh,
        overlap_thresh=overlap_thresh,
        dis_polar_thresh=dis_polar_thresh,
    )
    if season is not None:
        summary_comp = summary_comp.filter(
            pl.col("time")
            .is_in(pl.lit(season.implode().first(), pl.List(pl.Datetime("ms"))))
            .over("spell")
        )
    summary_comp = summary_comp.filter(pl.col("len") > 2).with_columns(slowness=slowness_expr())
    spells = (
        summary_comp
        .group_by("spell", maintain_order=True)
        .agg(
            pl.col("time"),
            pl.col("jet ID"),
            pl.col("lon_overlap"),
            pl.col("slowness"),
            pl.col("dis_polar"),
            pl.col("is_polar"),
            len=pl.len(),
            mean_is_polar=pl.col("is_polar").mean(),
            slowness_sum=pl.col("slowness").sum(),
        )
    )
    jet = pl.when(pl.col("mean_is_polar") < 0.5).then(pl.lit("STJ")).otherwise(pl.lit("EDJ"))
    result = spells.filter(pl.col("len") >= 20).group_by(jet).agg(pl.col("spell").n_unique())
    n_spells.append(result.with_columns(dist_thresh=pl.lit(dist_thresh)))

In [ ]:
# n_spells = pl.concat(n_spells)
n_spells_s = n_spells.filter(pl.col("literal") == "STJ")
n_spells_e = n_spells.filter(pl.col("literal") == "EDJ")
plt.plot(n_spells_s["dist_thresh"], n_spells_s["spell"])
plt.plot(n_spells_e["dist_thresh"], n_spells_e["spell"])

In [ ]:
_ = plt.hist(summer_filter.join(cross, on="time").filter(pl.col("dis_polar") < 0.12, pl.col("lon_overlap") > 0.7, pl.col("is_polar") > 0.6)["dist"], bins=51)
plt.figure()
_ = plt.hist(summer_filter.join(cross, on="time").filter(pl.col("dis_polar") < 0.12, pl.col("lon_overlap") > 0.7, pl.col("is_polar") < 0.4)["dist"], bins=51)


In [ ]:
_ = plt.hist(summary_comp.filter(pl.col("is_polar") < 0.4)["slowness"].replace(float("inf"), None).log(), bins=51)
plt.figure()
_ = plt.hist(spells.filter(pl.col("mean_is_polar") < 0.4)["slowness_sum"].replace(float("inf"), None).log(), bins=51)
plt.figure()
_ = plt.hist(spells.filter(pl.col("mean_is_polar") < 0.4)["len"], bins=np.arange(101) - 0.5)

In [ ]:
_ = plt.hist(summary_comp.filter(pl.col("is_polar") > 0.5)["slowness"].replace(float("inf"), None).log(), bins=51)
plt.figure()
_ = plt.hist(spells.filter(pl.col("mean_is_polar") > 0.5)["slowness_sum"].replace(float("inf"), None).log(), bins=51)
plt.figure()
_ = plt.hist(spells.filter(pl.col("mean_is_polar") > 0.5)["len"], bins=np.arange(50) - 0.5)

In [ ]:
spells_ = spells.filter(pl.col("mean_is_polar") < 0.4)
haha = spells_.group_by(pl.col("time").list.first().dt.ordinal_day()).agg(pl.col("slowness_sum").mean()).sort("time")
plt.plot(haha["time"], haha["slowness_sum"])
plt.figure()
spells_ = spells.filter(pl.col("mean_is_polar") > 0.6)
haha = spells_.group_by(pl.col("time").list.first().dt.ordinal_day()).agg(pl.col("slowness_sum").mean()).sort("time")
plt.plot(haha["time"], haha["slowness_sum"])

In [ ]:
spells_ = spells.filter(pl.col("mean_is_polar") < 0.4)
haha = spells_.group_by(pl.col("time").list.first().dt.ordinal_day()).agg(pl.col("len").mean()).sort("time")
plt.plot(haha["time"], haha["len"])
plt.figure()
spells_ = spells.filter(pl.col("mean_is_polar") > 0.6)
haha = spells_.group_by(pl.col("time").list.first().dt.ordinal_day()).agg(pl.col("len").mean()).sort("time")
plt.plot(haha["time"], haha["len"])

In [ ]:
from jetutils.plots import plot_seasonal
data_vars = [
    "mean_theta",
    "mean_lev",
    "pers",
    "slowness"
]
fig = plot_seasonal(
    props_catd,
    data_vars,
    nrows=2,
    ncols=2,
    clear=False,
    folder="Variability",
    suffix="_subset",
    numbering=True,
    save=False,
)

In [ ]:
filters_for_variables = {
    "APVO": ["cold", "warm"],
    "CPVO": ["cold", "warm"],
    "SAPVS": ["cold", "warm"],
    "SCPVS": ["cold", "warm"],
    "TAPVS": ["cold", "warm"],
    "TCPVS": ["cold", "warm"],
    "t2m": ["cold", "warm"],
    "tp": [
        "warm_entrance",
        "cold_exit",
        "cold",
        "warm",
        "warm_far_entrance",
    ],
    "theta": [
        "cold_exit",
        "warm_exit",
        "cold_entrance",
        "warm_entrance",
        "cold",
        "warm",
    ],
    "theta:grad": [
        "core",
    ],
    "PV330": [
        "cold_exit",
        "warm_exit",
        "cold_entrance",
        "warm_entrance",
        "cold",
        "warm",
    ],
    "PV330:grad": [
        "core",
    ],
    "PV350": [
        "cold_exit",
        "warm_exit",
        "cold_entrance",
        "warm_entrance",
        "cold",
        "warm",
    ],
    "PV350:grad": [
        "core",
    ],
    "EKE250": ["cold", "warm", "core"],
    "hor": ["cold", "warm", "core"],
}
reduce_dict = {var: pl.Expr.any for var in ["APVO", "CPVO", "SAPVS", "SCPVS", "TAPVS", "TCPVS"]}

In [ ]:
from jetutils.geospatial import prepare_last_step_1, prepare_last_step_2

pwe_path = basepath.joinpath("props_with_extras.parquet")
if not pwe_path.is_file():
    props_with_extras = prepare_last_step_1(basepath, filters_for_variables, phat_props, reduce_dict)
    props_with_extras.write_parquet(pwe_path)
else:
    props_with_extras = pl.read_parquet(pwe_path)

tss_path = Path(DATADIR, "ERA5/timeseries")
for f in tss_path.glob("*.parquet"):
    ts = pl.read_parquet(f)
    if len(f.stem.split("_")) == 2:
        ts = ts.rename({ts.columns[-1]: f.stem})
    else:
        ts = ts.pivot("region", index="time", column_naming="combine")
    props_with_extras = props_with_extras.join(ts, on="time", how="left")
props_with_extras_summer = summer_filter.join(props_with_extras, on="time")
grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")

everything = {}
for jet in ["STJ", "EDJ"]:
    everything[jet] = prepare_last_step_2(props_with_extras, spells_list[jet], summer, grams_wr=grams_wr, n_bootstraps=1000)

In [ ]:
clusters_da = np.abs(xr.open_dataarray(basepath.joinpath("cluster_df.nc")).load())
clusters_da = clusters_da.interp(lat=np.arange(32, 72, 0.5), method="nearest")

# plt.savefig(f"{FIGURES}/jet_persistence/regions.png")

region_T = compute_anomalies_pl(
    pl.read_parquet(basepath.joinpath("region_T_6H.parquet")), smooth_clim=31, other_index_columns=["region"], detrend=True
)
region_T = region_T.rolling("time", period="3d", group_by="region").agg(
    pl.col("t2m").mean()
)
region_T_ = summer_filter.join(region_T, on="time")
hws = get_spells(
    region_T_,
    pl.col("t2m") > pl.col("t2m").quantile(0.95),
    group_by=["region"],
    # fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=5),
).sort("region")
region_tp = compute_anomalies_pl(
    pl.read_parquet(basepath.joinpath("region_tp_6H.parquet")), smooth_clim=31, other_index_columns=["region"]
)
region_tp = region_tp.rolling("time", period="3d", group_by="region").agg(
    pl.col("tp").mean()
)
region_tp_ = summer_filter.join(region_tp, on="time")

pes = get_spells(
    region_tp_,
    pl.col("tp") > pl.col("tp").quantile(0.95),
    group_by=["region"],
    # fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=3),
).sort("region")
drs = get_spells(
    region_tp_,
    pl.col("tp") < pl.col("tp").quantile(0.05),
    group_by=["region"],
    # fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=5),
).sort("region")

# correlation, jet-centred

In [ ]:
from jetutils.definitions import to_expr

def pers_agg(a: str | pl.Expr, b: str | pl.Expr):
    a = to_expr(a).replace([float("nan"), float("inf"), -float("inf")], None)
    b = to_expr(b).replace([float("nan"), float("inf"), -float("inf")], None)
    a_ = a - a.mean()
    b_ = b - b.mean()
    cov = (a_ * b_).sum()
    return cov / (a.var() * b.var()).sqrt()

In [ ]:
for var in ["APVO", "CPVO", "PV330", "PV350", "theta", "tp", "t2m", "hor", "EKE250"]:
    df = pl.read_parquet(basepath.joinpath(f"{var}_relative.parquet"))
    df = summer_filter.join(df, on="time")
    props_ = summer_filter.join(props_catd["time", "jet", "slowness", "pers"], on="time")
    df = df.join(props_, on=["time", "jet"])

    huh = df.group_by("jet", "norm_index", "n", maintain_order=True).agg(
        pers_agg(f"{var}_interp", "pers").alias(var)
    )
    fig = plt.figure(figsize=(10, 6))
    polars_to_xarray(huh, ["jet", "n", "norm_index"]).plot(col="jet")
    

# quantile stuff

In [ ]:
from jetutils.definitions import to_expr
from typing import Sequence

def bin_by(df: pl.DataFrame, by: str | pl.Expr, jet: str, data_vars: Sequence[str], n_q: int = 100, lags: pl.Series | None = None):
    jet: pl.Expr = pl.col("jet") == jet
    qs = np.linspace(0, 1, n_q + 1).tolist()
    dq = qs[1] - qs[0]
    labels = [f"{int((q + dq / 2) * 100):d}" for q in qs]
    
    by = to_expr(by)
    filter_ = by.qcut(qs[1:], labels=labels, allow_duplicates=True).cast(pl.String()).str.to_integer()
    filter_ = df.filter(jet).drop_nulls(by).drop_nans(by).select("time", catd=filter_)
    if lags is None:
        lags = pl.Series("lags", [datetime.timedelta(0)])
    filter_ = filter_.join(lags.to_frame(), how="cross")
    filter_ = filter_.with_columns(pl.col("time") + pl.col("lags"))
    aggs = {var: pl.col(var).mean() for var in data_vars}
    aggs = {"len": pl.col("time").len()} | aggs
    return (
        df
        .join(filter_, on="time", how="left") 
        .group_by("catd", "jet", "lags")
        .agg(**aggs)
        .sort("catd", "lags", "jet")
    )

## forward

In [ ]:
from itertools import pairwise
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")
lags = pl.Series("lags", [datetime.timedelta(days=n.item()) for n in np.arange(-4, 5)])
n_q = 31
data_vars = [
    "mean_lat",
    "mean_theta",
    "mean_s",
    "wavinessR16",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(props_as_df_anoms, by, jet, data_vars=data_vars, n_q=n_q, lags=lags).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet, lags=datetime.timedelta(days=3))
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(ouais["catd"] / 100, ouais[data_var], label=PRETTIER_VARNAME[data_var], color=colors_props[k], lw=2)
        ax.set_title(f"Jet slowsness vs. props. of the {jet[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    "tp_tropical",
    "tp_gulfstream",
    "t2m_tropical",
    "t2m_east",
    "t2m_west",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(props_as_df_anoms, by, jet, data_vars=data_vars, n_q=n_q).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet)
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(ouais["catd"] / 100, ouais[data_var], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet slowsness vs. props. of the {jet[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [
    f"t2m_{i}" for i in range(1, 7)
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(props_as_df_anoms, by, jet, data_vars=data_vars, n_q=n_q, lags=pl.Series("lags", [datetime.timedelta(days=0)])).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet)
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(ouais["catd"] / 100, ouais[data_var], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [
    f"tp_{i}" for i in range(1, 7)
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(props_as_df_anoms, by, jet, data_vars=data_vars, n_q=n_q, lags=pl.Series("lags", [datetime.timedelta(days=0)])).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet)
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(ouais["catd"] / 100, ouais[data_var], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=2, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [
    "mean_lat",
    "tp-warm",
    "tp-cold",
    "t2m-warm",
    "t2m-cold",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(props_as_df_anoms, by, jet, data_vars=data_vars, n_q=n_q, lags=pl.Series("lags", [-datetime.timedelta(days=0)])).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet)
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(ouais["catd"] / 100, ouais[data_var], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet slowsness vs. props. of the {jet[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
data_vars = [
    # "tp-warm",
    # "tp-cold",
    # "t2m-warm",
    # "t2m-cold",
    "mean_lat",
    "mean_theta",
    "mean_s",
    "wavinessR16",
] + [
    f"t2m_{i}" for i in range(1, 7)
] + [
    f"tp_{i}" for i in range(1, 7)
]
by = "pers_catd"
n_q = 101
lags = pl.Series("lags", [datetime.timedelta(days=int(n)) for n in np.arange(-10, 11)])

In [ ]:
def the_struct(var):
    return pl.struct(
        at=pl.col("lags").get(pl.col(var).abs().arg_max()),
        val=pl.col(var).get(pl.col(var).abs().arg_max()),
    )
for jet in ["STJ", "EDJ"]:
    huh = bin_by(props_as_df_anoms, by, jet, data_vars=data_vars, n_q=n_q, lags=lags).drop_nulls("catd").filter(pl.col("jet") == jet)
    res = huh.group_by("lags", maintain_order=True).agg(
        **{
            data_var: pds.lin_reg_report("catd", target=data_var)
            .struct.field("beta")
            .first() * 100
            for data_var in data_vars
        }
    ).select(**{var: the_struct(var) for var in data_vars})
    print(res)

## backwards
quantile of the prop vs absolute value of slowness

In [ ]:
from itertools import pairwise
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    "mean_lat",
    "mean_theta",
    "mean_s",
    "wavinessR16",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(["pers", data_var]), data_var, jet, data_vars=["pers"], n_q=n_q).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet)
            ax.plot(ouais["catd"] / 100, ouais["pers"], label=PRETTIER_VARNAME[data_var], color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [
    "tp_tropical",
    "tp_gulfstream",
    "t2m_tropical",
    "t2m_east",
    "t2m_west",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(["pers", data_var]), data_var, jet, data_vars=["pers"], n_q=n_q).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet)
            ax.plot(ouais["catd"] / 100, ouais["pers"], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    f"t2m_{i}" for i in range(1, 7)
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(["pers", data_var]), data_var, jet, data_vars=["pers"], n_q=n_q).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet)
            ax.plot(ouais["catd"] / 100, ouais["pers"], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    f"tp_{i}" for i in range(1, 7)
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(["pers", data_var]), data_var, jet, data_vars=["pers"], n_q=n_q).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet)
            ax.plot(ouais["catd"] / 100, ouais["pers"], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise
props_as_df_anoms = compute_anomalies_pl(props_with_extras, other_index_columns=["jet"], standardize=True)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    "mean_lat",
    "tp-warm_entrance",
    "tp-cold",
    "t2m-warm",
    "t2m-cold",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(len(bys), 2, figsize=(10, 5), sharex="all", sharey="row", gridspec_kw={"wspace": 0.1, "left": 0.1})
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0., color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(["pers", data_var]), data_var, jet, data_vars=["pers"], n_q=n_q).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet)
            ax.plot(ouais["catd"] / 100, ouais["pers"], label=PRETTIER_VARNAME.get(data_var, data_var), color=colors_props[k], lw=2)
        ax.set_title(f"Jet persistence vs. props. of the {jet[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

# jet pos

In [ ]:
jet_pos_da = jet_position_as_da(jets)

plt.ion()
MYPURPLES = LinearSegmentedColormap.from_list(
    "mypurples", ["#ffffff", COLORS_EXT[4], COLORS_EXT[5]]
)
MYPINKS = LinearSegmentedColormap.from_list(
    "mypinks", ["#ffffff", COLORS_EXT[7], COLORS_EXT[8]]
)

to_plot_1 = []
to_plot_2 = []
for month in range(1, 13):
    this_da = jet_pos_da.sel(time=jet_pos_da.time.dt.month == month)
    to_plot_1.append((this_da < 0.5).mean("time"))
    to_plot_2.append((this_da > 0.5).mean("time"))

clu = Clusterplot(3, 4, get_region(jet_pos_da), row_height=3.4)
im, kwargs = clu.add_contourf(
    [tp2 * 100 for tp2 in to_plot_2],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.gothic_r,
    draw_cbar=False,
    titles=MONTH_NAMES,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Eddy-driven jet occurence [$\%$]", fontsize=13)
im, kwargs = clu.add_contourf(
    [tp1 * 100 for tp1 in to_plot_1],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.flamingo_r,
    draw_cbar=False,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Subtropical jet occurence [\%]", fontsize=13)
clu.resize_relative([1.0, 1.06])

In [ ]:
jet_pos_da_eps = jet_position_as_da(
    spells_list["EDJ"]
    .join(jets.cast({"time": pl.Datetime("ms")}), on="time")
    .drop("spell", "relative_index")
)
plt.ion()
MYPURPLES = LinearSegmentedColormap.from_list(
    "mypurples", ["#ffffff", COLORS_EXT[4], COLORS_EXT[5]]
)
MYPINKS = LinearSegmentedColormap.from_list(
    "mypinks", ["#ffffff", COLORS_EXT[7], COLORS_EXT[8]]
)

to_plot_1 = []
to_plot_2 = []
for month in range(7, 10):
    this_da = jet_pos_da_eps.sel(time=jet_pos_da_eps.time.dt.month == month)
    to_plot_1.append((this_da < 0.5).mean("time"))
    to_plot_2.append((this_da > 0.5).mean("time"))

clu = Clusterplot(1, 3, get_region(jet_pos_da_eps), row_height=6)
im, kwargs = clu.add_contourf(
    [tp2 * 100 for tp2 in to_plot_2],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.gothic_r,
    draw_cbar=False,
    titles=MONTH_NAMES[6:],
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Eddy-driven jet occurence [$\%$]", fontsize=13)
im, kwargs = clu.add_contourf(
    [tp1 * 100 for tp1 in to_plot_1],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.flamingo_r,
    draw_cbar=False,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Subtropical jet occurence [\%]", fontsize=13)
clu.resize_relative([1.0, 1.06])

# indiv spells

TODO:
- grouping props?
- add wrs

LATER:
- plot_interp for selection of jet-centred averaged props

In [ ]:
relevant_keys_STJ = [
    "theta-cold-STJ.during",
    "theta-cold-STJ.before",
    "theta-warm-STJ.during",
    "theta-warm-STJ.before",
    "theta:grad-core-STJ.during",
    "theta:grad-core-STJ.before",
    "PV350-cold-EDJ.during",
    "PV350-cold-EDJ.before",
    "PV350-warm-EDJ.during",
    "PV350-warm-EDJ.before",
    "t2m-cold-STJ.during",
    "t2m-cold-STJ.before",
    "t2m-warm-STJ.during",
    "t2m-warm-STJ.before",
    "mean_lat-STJ.during",
    "mean_lat-STJ.before",
    "tp-warm_far_entrance-STJ.during",
    "tp-warm_far_entrance-STJ.before",
    "SAPVS-warm-STJ.during",
    "SAPVS-warm-STJ.before",
    "SAPVS-cold-STJ.during",
    "SAPVS-cold-STJ.before",
    "SCPVS-warm-STJ.during",
    "SCPVS-warm-STJ.before",
    "SCPVS-cold-STJ.during",
    "SCPVS-cold-STJ.before",
    "TAPVS-warm-STJ.during",
    "TAPVS-warm-STJ.before",
    "TAPVS-cold-STJ.during",
    "TAPVS-cold-STJ.before",
    "TCPVS-warm-STJ.during",
    "TCPVS-warm-STJ.before",
    "TCPVS-cold-STJ.during",
    "TCPVS-cold-STJ.before",
]

relevant_keys_EDJ = [
    "theta-cold-EDJ.during",
    "theta-cold-EDJ.before",
    "theta-warm-EDJ.during",
    "theta-warm-EDJ.before",
    "PV330-cold-EDJ.during",
    "PV330-cold-EDJ.before",
    "PV330-warm-EDJ.during",
    "PV330-warm-EDJ.before",
    "PV330-warm_exit-EDJ.during",
    "PV330-warm_exit-EDJ.before",
    "theta:grad-core-EDJ.during",
    "theta:grad-core-EDJ.before",
    "t2m-cold-EDJ.during",
    "t2m-cold-EDJ.before",
    "t2m-warm-EDJ.during",
    "t2m-warm-EDJ.before",
    "mean_lat-EDJ.during",
    "mean_lat-EDJ.before",
    "tp-warm_entrance-EDJ.during",
    "tp-warm_entrance-EDJ.before",
    "SAPVS-warm-EDJ.during",
    "SAPVS-warm-EDJ.before",
    "SAPVS-cold-EDJ.during",
    "SAPVS-cold-EDJ.before",
    "SCPVS-warm-EDJ.during",
    "SCPVS-warm-EDJ.before",
    "SCPVS-cold-EDJ.during",
    "SCPVS-cold-EDJ.before",
    "TAPVS-warm-EDJ.during",
    "TAPVS-warm-EDJ.before",
    "TAPVS-cold-EDJ.during",
    "TAPVS-cold-EDJ.before",
    "TCPVS-warm-EDJ.during",
    "TCPVS-warm-EDJ.before",
    "TCPVS-cold-EDJ.during",
    "TCPVS-cold-EDJ.before",
]

relevant_keys_STJ = [key for key in relevant_keys_STJ if key.split(".")[-1] == "during"]
relevant_keys_EDJ = [key for key in relevant_keys_EDJ if key.split(".")[-1] == "during"]

In [ ]:
from matplotlib.cm import ScalarMappable
plot_kwargs = {
    "STJ": (
        relevant_keys_STJ, 
        (
            pl.col("is_early_or_late"),
            pl.col("tp-warm_far_entrance-STJ.during") > 0.0,
        ),
        pl.col("mean_lat-STJ.during"),
    ),
    "EDJ": (
        relevant_keys_EDJ, 
        (
            pl.col("tp-warm_entrance-EDJ.during") > 0., 
            (pl.col("TAPVS-cold-EDJ.during") > 0.) | (pl.col("TCPVS-cold-EDJ.during") > 0.),
        ),
        pl.col("mean_lat-EDJ.during"),
    ),
}
groupnames_stj = [
    r"\textbf{Early/late, low precip}", 
    r"\textbf{Early/late, high precip}",
    r"\textbf{Mid summer, low precip}",
    r"\textbf{Mid summer, high precip}",
]
groupnames_edj = [
    r"\textbf{Low precip, no CWB}", 
    r"\textbf{Low precip, CWB}",
    r"\textbf{High precip, no CWB}",
    r"\textbf{High precip, CWB}",
]
groupnames = {"STJ": groupnames_stj, "EDJ": groupnames_edj}
cmap = colormaps.BlWhRe
norm = BoundaryNorm([-2, -1, -0.5, 0.5, 1, 2], cmap.N)
im = ScalarMappable(norm, cmap=cmap)
colors_groups = colormaps.pastel([2, 3, 4, 6, 7, 8, 0, 5, 1])

for jet, values in plot_kwargs.items():
    keys, grouping, sorting = values
    pvals_keys = [f"{key}.pvals" for key in keys]
    df: pl.DataFrame = everything[jet]
    groups = df.select("spell", group=pl.concat_str(*[g.cast(pl.UInt8()) for g in grouping]).str.to_integer(base=2))
    df = df.join(groups, on="spell", how="left").sort(
        pl.col("spell") == -2, 
        pl.col("spell") == -1, 
        pl.col("group"), 
        sorting
    )
    groups = df.select("spell", "group")
    to_plot = df.select(*keys).to_numpy().T
    pvals = df.select(*pvals_keys).to_numpy().T
    to_plot = np.where(pvals < .05, to_plot, np.nan)
    
    ## main
    fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"aspect": "equal"})
    x = np.arange(to_plot.shape[1])
    y = np.arange(to_plot.shape[0])
    ax.pcolormesh(x, y, to_plot, norm=norm, cmap=cmap, edgecolor="black", lw=.5)
    fig.colorbar(im, ax=ax, shrink=0.56, pad=0.01)
    
    ## yticks
    _ = ax.set_yticks(y, labels=keys)
    
    ## xticks
    spell_labels = []
    spell_labels_colors = []
    for i_spell, group in groups.filter(pl.col("spell") >= 0).iter_rows():
        spell_dates = spells_list[jet].filter(pl.col("spell") == i_spell)["time"]
        year = str(spell_dates.dt.year().mode().item()).zfill(4)
        first_date, last_date = (
            spell_dates.dt.strftime("%d/%m").gather([0, -1]).to_list()
        )
        spell_labels.append(r"$\mathbf{" + f"{year}, {first_date}-{last_date}" + r"}$")
        spell_labels_colors.append(colors_groups[group])

    spell_labels.append("Episode mean")
    spell_labels.append("Mean clim")
    spell_labels_colors.extend(["black", "black"])
    
    ax.set_xticks(
        x, labels=spell_labels, rotation=90
    )
    for color, ticklabel in zip(spell_labels_colors, ax.xaxis.get_ticklabels()):
        ticklabel.set_color(color)
        ticklabel.set_color(color)
    ax.set_xlabel(f"Persistent episode of the {jet}")
    
    yticklabels = []
    yticklabelcolors = []
    for key in keys:
        rest, when = key.split(".")
        elements = rest.split("-")
        ofthe = elements[-1]
        varname = elements[0]
        label = PRETTIER_VARNAME.get(varname, varname)
        if ":" in varname and varname.split(":")[-1] == "grad":
            varname_ = varname.split(":")[0]
            label = f"grad of {PRETTIER_VARNAME.get(varname_, varname_)}"
        if when == "during":
            yticklabelcolors.append("black")
        else:
            yticklabelcolors.append("lightslategrey")
        if len(elements) == 3:
            where_ = elements[1]
            where_ = ", ".join(where_.split("_"))
            label = f"{label}, {where_}"
        yticklabels.append(label)
    
    ax.set_yticks(y, labels=yticklabels)
    for color, ticklabel in zip(yticklabelcolors, ax.yaxis.get_ticklabels()):
        ticklabel.set_color(color)
        
    x_anchor = -0.1
    y_anchor = -0.15
    height_per_height = .05
    toptext = y_anchor + (len(groupnames[jet]) + 0.4) * height_per_height
    width = .215
    height = (len(groupnames[jet]) + 1) * height_per_height + 0.01
    offset = 0.025 * float(jet == "EDJ")
    phax = fig.add_axes([x_anchor + offset, y_anchor, width - offset, height])
    phax.xaxis.set_visible(False)
    phax.yaxis.set_visible(False)
    phax.set_zorder(1000)
    phax.patch.set_alpha(0.)
    phax.patch.set_color('w')
    fig.text(x_anchor + offset + 0.01, toptext, r"\textbf{Groups}:", color="black", fontsize=20)
    txtkwargs = {"fontsize": 16}
    for i, groupname in enumerate(groupnames[jet]):
        fig.text(x_anchor + offset + 0.01, toptext - (i + 1) * height_per_height, groupname, color=colors_groups[i], **txtkwargs)
# keys = [key for key in relevant_keys_EDJ if key.split(".")[-1] == "during"]
# to_plot_ = everything["EDJ"].sort("mean_lat_EDJ.during").select(*keys)
# pvals_EDJ = [f"{key}.pvals" for key in keys]
# pvals = everything["EDJ"].sort("mean_lat_EDJ.during").select(*pvals_EDJ)
# to_plot = to_plot_.to_numpy().T[::-1, ::-1]
# to_plot = np.where(pvals.to_numpy().T < 1., to_plot_.to_numpy().T, np.nan)[::-1, ::-1]



In [ ]:
    # """WRS"""
    # regimes = df[order_x_, f"regime_during"].fill_null(0).cast(pl.Int32()).to_numpy()
    # regime_names = winner_names["name"][regimes].to_numpy()
    # cmap = colormaps.bold
    # cmap.set_under("none")
    # cmap.set_bad("none")
    # axes["X"].pcolormesh(
    #     x,
    #     height_ratios.sum() + [0, 1],
    #     regimes[None, :],
    #     cmap=cmap,
    #     norm=BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N),
    #     linewidth=0.5,
    #     edgecolor="white",
    # )
    # for x_, text_ in zip(x[:-1] + 0.5, regime_names):
    #     if text_ == "No":
    #         text_ = ""
    #     else:
    #         text_ = r"\textbf{" + text_ + r"}"
    #     axes["X"].text(
    #         x_,
    #         height_ratios.sum() + 0.44,
    #         text_,
    #         ha="center",
    #         va="center",
    #         fontsize=8,
    #         color="white" if text_ == "GB" else "black",
    #     )
    # labels = [a[1] for a in relevant_keys.values()]
    # labels.append("Weather Regime")


    # axes["X"].set_yticks(np.arange(len(labels)) + 0.5, labels=labels)

    # labels = axes["X"].get_yticklabels()
    # for key, label in zip(relevant_keys, labels):
    #     if key.split("_")[-1] == "before":
    #         label.set_color("darkslategrey")
    # for ypos in where_lines:
    #     axes["X"].axhline(ypos, color="black")

    # plt.draw()
    # tb = fig.get_tightbbox(fig.canvas.get_renderer())
    # fig.set_size_inches(tb.width, tb.height)
    # plt.draw()

    # figW, figH = fig.get_size_inches()
    # x0, _, w, _ = axes["X"].get_position().bounds
    # end_of_X = x0 + w
    # start_of_a = axes["A"].get_position().bounds[0]
    # extra_padding = start_of_a - end_of_X
    # new_W = figW * (1 - extra_padding)
    # fig.set_size_inches(new_W, figH)
    # fig.savefig(f"{FIGURES}/Persistence/last_v6_{jet}_{n_spells}spells.pdf")
    # plt.plot()

# interp

In [ ]:
variables = [
    "theta:clim",
    'theta:anom',
    'theta:clim_grad',
    'theta:anom_grad',
    'PV:clim',
    'PV:anom',
    'PV:clim_grad',
    'PV:anom_grad',
    'EKE250:clim',
    'EKE250:anom',
    'hor:clim',
    'hor:anom',
    't2m:clim',
    't2m:anom',
    'tp:clim',
    'tp:anom',
]
for var in ["SAPVS", "TAPVS", "SCPVS", "TCPVS", "APVO", "CPVO"]:
    variables.append(f"{var}:clim")
    variables.append(f"{var}:anom")

In [ ]:
create_all_relative_plots(spells_list, variables, basepath, odir=Path(FIGURES, "Persistence/figure_data"), season=summer, n_bootstraps=100)

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/RWB")
odir.mkdir(exist_ok=True)
variables = {}

for varname in ["APVO", "CPVO"]:
    variables[f"{varname}:clim"] = [7, colormaps.cet_l_bmy_r, [0, 60]]
    variables[f"{varname}:anom"] = [12, colormaps.BlWhRe, [-20, 20]]
for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/ground")
odir.mkdir(exist_ok=True)
variables = {
    "t2m:clim": [6, colormaps.bilbao_r, [280, 300]],
    "t2m:anom": [8, colormaps.BlWhRe, [-4, 4]],
    "tp:clim": [9, colormaps.freeze_r, [0, 6]],
    "tp:anom": [9, colormaps.brbg, [-2, 2]],
}

for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
        alpha=0.05
    )
    fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/upper_level")
odir.mkdir(exist_ok=True)

variables = {
    "theta:clim": [8, colormaps.bilbao_r, [330, 370]],
    "theta:anom": [8, colormaps.BlWhRe, [-6, 6]],
    "theta:clim_grad": [8, colormaps.bilbao_r, [0, 40]],
    "theta:anom_grad": [10, colormaps.BlWhRe, [-12, 12]],
    "PV:clim": [9, WERNLI_FLAIR, [0, 10]],
    "PV:anom": [8, colormaps.BlWhRe, [-1.2, 1.2]],
    "PV:clim_grad": [10, colormaps.bilbao_r, [0, 10]],
    "PV:anom_grad": [10, colormaps.BlWhRe, [-4, 4]],
}
for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eddies")
odir.mkdir(exist_ok=True)

variables = {
    "EKE250:clim": [9, colormaps.cet_l_bmy_r, [0, 200]],
    "EKE250:anom": [9, colormaps.BlWhRe, [-30, 30]],
    "hor:clim": [12, colormaps.BlWhRe, [-4, 4]],
    "hor:anom": [12, colormaps.BlWhRe, [-4, 4]],
}
for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

# when spells

### full grid

In [ ]:
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) - 0.3, f"{year}", fontsize=10)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells.pdf")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 4), gridspec_kw=dict(wspace=0, hspace=0))

ax = axes[0]
colors = [COLORS[2], COLORS[1]]
keys = list(spells_list)
i = 1
otheralpha = 0.5
dys = {"EDJ": 40, "STJ": 40}
for key, val in spells_list.items():
    dy = dys[key]
    huh = (
        summer.dt.ordinal_day()
        .unique()
        .sort()
        .to_frame()
        .with_columns(
            time_2=pl.datetime(year=1959, month=1, day=1) + pl.duration(days="time")
        )
    )
    to_plot = gaussian_kde(val["time"].dt.ordinal_day(), bw_method=0.1).evaluate(
        huh["time"]
    )
    ax.fill_between(
        huh["time_2"],
        i,
        i + dy * to_plot,
        color=colors[1 - i],
        facecolor="none",
        alpha=1.0,
        lw=2,
    )
    ax.fill_between(
        huh["time_2"],
        i,
        i + dy * to_plot,
        color=colors[1 - i],
        alpha=otheralpha,
    )
    i = i - 1
ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
ax.xaxis.set_tick_params(labelsize=11, rotation=30)
ticks = ax.get_xticks()
ticklabels = ax.get_xticklabels()
ax.set_xticks(ticks, labels=[t.get_text() for t in ticklabels], ha="right")
ax.set_yticks([0.35, 1.5], labels=keys[::-1])
ax.yaxis.set_tick_params(length=0)
ax.set_title("a) Episode distrib.")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])

ax = axes[1]
dy = 1.
bins = np.logspace(-3.5, -0.5, 101)
summer_pers = phat_props_summer.select("time", "jet", "slowness_catd")
y1 = summer_pers.filter(pl.col("jet") == "STJ")["slowness_catd"]
y2 = summer_pers.filter(pl.col("jet") == "EDJ")["slowness_catd"]
for i, y in enumerate([y1, y2]):
    y_smo = gaussian_kde(
        np.log10(y.replace((0., float("inf")), None).drop_nulls()), bw_method=0.2
    ).evaluate(np.log10(bins))
    y_smo = (1 - i) + dy * y_smo
    ax.fill_between(bins, (1 - i), y_smo, color=COLORS[2 - i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[2 - i], lw=2)
ax.set_xscale("log")
ax.set_yticks([])
ax.set_title("b) Slowness [s/m]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])

ax = axes[2]
dy = .5
bins = np.logspace(-1.5, 0.5, 101)
summer_pers = phat_props_summer.select("time", "jet", "pers_catd")
y1 = summer_pers.filter(pl.col("jet") == "STJ")["pers_catd"]
y2 = summer_pers.filter(pl.col("jet") == "EDJ")["pers_catd"]
for i, y in enumerate([y1, y2]):
    y_smo = gaussian_kde(
        np.log10(y.replace((0., float("inf")), None).drop_nulls()), bw_method=0.2
    ).evaluate(np.log10(bins))
    y_smo = (1 - i) + dy * y_smo
    ax.fill_between(bins, (1 - i), y_smo, color=COLORS[2 - i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[2 - i], lw=2)
ax.set_xscale("log")
ax.set_yticks([])
ax.set_title("c) Persistence [s/m]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])


ax = axes[3]
dy = 1.3
bins = np.arange(4, 12, 0.15) - 0.125
for i, jet in enumerate(["EDJ", "STJ"]):
    y = spells_list[jet].group_by("spell").agg(pl.col("len").first())["len"] / 4
    y_smo = gaussian_kde(y, bw_method=0.3).evaluate(bins)
    y_smo = i + dy * y_smo
    ax.fill_between(bins, i, y_smo, color=COLORS[1 + i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[1 + i], lw=2)
ax.set_yticks([0, 0.5, 1, 1.5], labels=[str(i) for i in [0, 0.5] * 2])
ax.tick_params("y", left=False, right=True, labelleft=False, labelright=True)
ax.set_title("c) Episode length [day]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])
fig.savefig(f"{FIGURES}/Persistence/stats.pdf")

### with extremes

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
regions_I_want = [2, 3]
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
for region_name, color, ir in zip(DUNCANS_REGIONS_NAMES, colors_regions, range(1, 7)):
    if ir not in regions_I_want:
        continue
    ax.plot([], [], color=color, lw=3, label=region_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + len(regions_I_want) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) + len(regions_I_want) - 0.3, f"{year}", fontsize=10)
    for j, ir in enumerate(regions_I_want):
        region_name = DUNCANS_REGIONS_NAMES[ir]
        color = colors_regions[ir - 1]
        j = j + len(spells_list)
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=color,
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = hws.filter(pl.col("time").dt.year() == year, pl.col("region") == ir)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=color, lw=3)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells+hw.pdf")

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
regions_I_want = [2, 3]
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
for region_name, color, ir in zip(DUNCANS_REGIONS_NAMES, colors_regions, range(1, 7)):
    if ir not in regions_I_want:
        continue
    ax.plot([], [], color=color, lw=3, label=region_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + len(regions_I_want) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) + len(regions_I_want) - 0.3, f"{year}", fontsize=10)
    for j, ir in enumerate(regions_I_want):
        region_name = DUNCANS_REGIONS_NAMES[ir]
        color = colors_regions[ir - 1]
        j = j + len(spells_list)
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=color,
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = pes.filter(pl.col("time").dt.year() == year, pl.col("region") == ir)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=color, lw=3)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells+pe.pdf")

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
regions_I_want = [2, 3]
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
for region_name, color, ir in zip(DUNCANS_REGIONS_NAMES, colors_regions, range(1, 7)):
    if ir not in regions_I_want:
        continue
    ax.plot([], [], color=color, lw=3, label=region_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + len(regions_I_want) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) + len(regions_I_want) - 0.3, f"{year}", fontsize=10)
    for j, ir in enumerate(regions_I_want):
        region_name = DUNCANS_REGIONS_NAMES[ir]
        color = colors_regions[ir - 1]
        j = j + len(spells_list)
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=color,
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = drs.filter(pl.col("time").dt.year() == year, pl.col("region") == ir)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=color, lw=3)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells+dr.pdf")

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
figure = plt.figure(figsize=(10, 4.5), constrained_layout=True)
subfigs = figure.subfigures(1, 2, wspace=0.00, width_ratios=[6.5, 3.5])
n_bootstraps = 1000
with plt.style.context(["seaborn-v0_8-darkgrid", STYLE_SHEET]):
    fig = subfigs[0]
    axes = fig.subplots(2, 2, sharex="col", sharey="row")
    axes = axes.T
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = spells_list[spell_of].drop("jet ID")
        spells = extend_spells(spells, time_before=datetime.timedelta(days=4))
        times = create_bootstrapped_times(spells, summer, n_bootstraps)
        for j, (name, df) in enumerate({"t2m": region_T, "tp": region_tp}.items()):
            ax = axes[i, j]
            region_ts_masked = times.join(df, on="time").sort("sample_index", "region", "spell", "relative_index")
            x, alive_spells = (
                region_ts_masked
                .filter(pl.col("region") == 1, pl.col("sample_index") == n_bootstraps)
                .group_by("relative_index")
                .agg(pl.col("spell").n_unique())
                .sort("relative_index")
                .to_numpy()
                .T
            )
            region_ts_masked = region_ts_masked.group_by("sample_index", "region", "relative_index", maintain_order=True).agg(pl.col(name).mean())
            pvals = region_ts_masked.group_by("region", "relative_index", maintain_order=True).agg(pl.col(name).rank().last() / n_bootstraps)
            pvals = pvals.with_columns(pl.min_horizontal(pl.col(name), 1 - pl.col(name)) * 2)
            region_ts_masked = region_ts_masked.group_by("region", "relative_index", maintain_order=True).agg(pl.col(name).last())
            
            x = x / 4
            filter_ = alive_spells > 15
            ax.axhline(0, color="black")
            ax.axvline(0, color="black")
            for reg, huh_ in region_ts_masked.group_by("region", maintain_order=True):
                y = huh_[name].to_numpy()
                y = y * 1000 if name == "tp" else y
                reg = reg[0]
                pvals_ = pvals.filter(pl.col("region") == reg)
                pvals_ = pvals_[name].to_numpy()
                ax.plot(
                    x[filter_],
                    y[filter_],
                    color=colors_regions[reg - 1],
                    label=DUNCANS_REGIONS_NAMES[reg - 1],
                    lw=1,
                )
                ax.plot(
                    np.where(pvals_[filter_] < 0.1, x[filter_], np.nan),
                    np.where(pvals_[filter_] < 0.1, y[filter_], np.nan),
                    color=colors_regions[reg - 1],
                    label=DUNCANS_REGIONS_NAMES[reg - 1],
                    lw=3,
                )
            k = 2 * j + i
            ax.annotate(
                f"{ascii_lowercase[k]})",
                xy=(0, 1),
                xycoords="axes fraction",
                xytext=(+0.5, -0.5 - 6.5 * float(k == 0)),
                textcoords="offset fontsize",
                fontsize="medium",
                verticalalignment="top",
                fontfamily="serif",
                bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
            )
    # axes[0, 0].legend(ncol=2, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)
    axes[0, 0].set_ylabel(PRETTIER_VARNAME["t2m"] + " [K]")
    axes[0, 1].set_ylabel(PRETTIER_VARNAME["tp"] + " [mm]")
    axes[0, 0].set_title("STJ episodes")
    axes[1, 0].set_title("EDJ episodes")
    axes[0, 1].set_xlabel("Time around onset [day]")
    axes[1, 1].set_xlabel("Time around onset [day]")

clu = Clusterplot(1, 1, (-10, 40, 35, 72), row_height=5, fig=subfigs[1])
cmap = colormaps.pastel
ax = clu.axes[0]
unique_clusters = np.arange(1, 7)
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
clusters_da.assign_coords(lon=clusters_da.lon - clu.central_longitude).plot(
    ax=ax, cmap=cmap, norm=norm, add_colorbar=False, add_labels=False
)
for j in range(6):
    lo = (
        clusters_da.lon.where(clusters_da == (j + 1)).mean().item()
        - j
        - 3 * (j == 0)
        + 2 * (j == 2)
        + 3 * (j == 1)
        - (j == 4)
        - clu.central_longitude
    )
    la = clusters_da.lat.where(clusters_da == (j + 1)).mean().item() - (j == 5) * 2
    color = "black"
    ax.text(
        lo,
        la,
        DUNCANS_REGIONS_NAMES[j],
        ha="center",
        va="center",
        fontweight="bold",
        color=color,
        fontsize=12,
    )

ax.annotate(
    f"{ascii_lowercase[k + 1]})",
    xy=(float(k == 0), 1),
    xycoords="axes fraction",
    xytext=(+0.5 - 2 * float(k == 0), -0.5),
    textcoords="offset fontsize",
    fontsize="medium",
    verticalalignment="top",
    fontfamily="serif",
    bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
)
figure.savefig(f"{FIGURES}/Persistence/region_timeseries_30spells.pdf")

In [ ]:
alors = extend_spells(spells_list[spell_of].drop("jet ID"), time_before=datetime.timedelta(days=4)).join(timeseries["t2m_west"], on="time")

fig, ax = plt.subplots(figsize=(25, 10))
for _, mhh in alors.group_by("spell"):
    ax.plot(mhh["t2m"])
ax.plot(alors.group_by("relative_time", maintain_order=True).agg(pl.col("t2m").mean())["t2m"], lw=4)

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
timeseries = {}
colors = {}
for i, f in enumerate(Path(DATADIR, "ERA5/timeseries").glob("t*.parquet")):
    timeseries[f.stem] = pl.read_parquet(f)
    subname = f.stem.split("_")[0]
    timeseries[f.stem] = timeseries[f.stem].group_by(pl.col("time").dt.round("1d")).agg(pl.col(subname).mean()).sort("time")
    colors[f.stem] = colors_regions[i]
n_bootstraps = 1000
with plt.style.context(["seaborn-v0_8-darkgrid", STYLE_SHEET]):
    fig, axes = plt.subplots(2, 2, sharex="col", sharey="row")
    axes = axes.T
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = daily_spells_list[spell_of].drop("jet ID")
        spells = extend_spells(spells, time_before=datetime.timedelta(days=4))
        times = create_bootstrapped_times(spells, summer_daily, n_bootstraps)
        for name, df in timeseries.items():
            subname, region = name.split("_")
            j = int(subname == "tp")
            ax = axes[i, j]
            ts_masked = times.join(df, on="time").sort("sample_index", "spell", "relative_index")
            x, alive_spells = (
                ts_masked
                .filter(pl.col("sample_index") == n_bootstraps)
                .group_by("relative_index")
                .agg(pl.col("spell").n_unique())
                .sort("relative_index")
                .to_numpy()
                .T
            )
            season_mean = ts_masked[subname].mean()
            ts_masked = ts_masked.group_by("sample_index", "relative_index", maintain_order=True).agg(pl.col(subname).mean())
            pvals = ts_masked.group_by("relative_index", maintain_order=True).agg(pl.col(subname).rank().last() / n_bootstraps)
            pvals = pvals.with_columns(pl.min_horizontal(pl.col(subname), 1 - pl.col(subname)) * 2)
            ts_masked = ts_masked.group_by("relative_index", maintain_order=True).agg(pl.col(subname).last())
            
            x = x / 4
            filter_ = alive_spells > 15
            ax.axhline(0, color="black")
            ax.axvline(0, color="black")
            y = ts_masked[subname].to_numpy()
            y = y * 1000 if subname == "tp" else y
            season_mean = season_mean * 1000 if subname == "tp" else season_mean
            pvals_ = pvals[subname].to_numpy()
            ax.plot(
                x[filter_],
                y[filter_],
                color=colors[name],
                # label=region,
                lw=1,
            )
            if subname == "tp":
                ax.plot(
                    x[filter_][[0, -1]],
                    [season_mean, season_mean],
                    color=colors[name],
                    ls="dashed",
                    lw=1,
                )
            ax.plot(
                np.where(pvals_[filter_] < 0.1, x[filter_], np.nan),
                np.where(pvals_[filter_] < 0.1, y[filter_], np.nan),
                color=colors[name],
                label=region,
                lw=3,
            )
            k = 2 * j + i
            ax.annotate(
                f"{ascii_lowercase[k]})",
                xy=(0, 1),
                xycoords="axes fraction",
                xytext=(+0.5, -0.5 - 6.5 * float(k > 0)),
                textcoords="offset fontsize",
                fontsize="medium",
                verticalalignment="top",
                fontfamily="serif",
                bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
            )
    axes[1, 0].legend(ncol=2, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)
    axes[0, 1].legend(ncol=1, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)

    axes[0, 0].set_ylabel(PRETTIER_VARNAME["t2m"] + " [K]")
    axes[0, 1].set_ylabel(PRETTIER_VARNAME["tp"] + " [mm]")
    axes[0, 0].set_title("STJ episodes")
    axes[1, 0].set_title("EDJ episodes")
    axes[0, 1].set_xlabel("Time around onset [day]")
    axes[1, 1].set_xlabel("Time around onset [day]")

In [ ]:
from matplotlib.colors import hex2color

names = ["Cyan", "Yellow", "Orange", "Purple", "Green", "Blue"]
names = [f"{name}Pastel" for name in names]
base = r"\definecolor{"
middle = r"}{rgb}{"
end = r"}"
for color, name in zip(cmap(norm(np.arange(1, 7))), names):
    color = [f"{spec:.3f}" for spec in color[:3]]
    print(base + f"{name}" + middle + " ".join(color) + end)

name = "STJcolor"
color = hex2color(COLORS[2])
color = [f"{spec:.3f}" for spec in color[:3]]
print(base + f"{name}" + middle + " ".join(color) + end)

name = "EDJcolor"
color = hex2color(COLORS[1])
color = [f"{spec:.3f}" for spec in color[:3]]
print(base + f"{name}" + middle + " ".join(color) + end)

base = r"\textcolor{"
middle = r"}{\textbf{"
end = r"}}"
schema = []
for color, name in zip(names, DUNCANS_REGIONS_NAMES):
    schema.append(base + color + middle + name + end)
print(" & ".join(schema))

In [ ]:
for key, df in {"hw": hws, "pe": pes, "dr": drs}.items():
    out = summer.to_frame()

    for region in range(1, 7):
        alias = f"reg{region}"
        to_join = (
            df[["region", "time"]]
            .with_columns((pl.col("region") == region).alias(alias))
            .filter(alias)
            .drop("region")
        )
        out = out.join(to_join, on="time", how="left")
    for spell_of in ["STJ", "EDJ"]:
        spells = spells_list[f"{spell_of}"]
        spells = extend_spells(spells, time_before=datetime.timedelta(hours=6))
        spells = (
            spells[["time", "spell"]]
            .with_columns(spell=pl.lit(True))
            .rename({"spell": f"spell_{spell_of}"})
        )
        out = out.join(spells, on="time", how="left")
    out = out.fill_null(False).fill_nan(False)

    aggs = {}
    for i, spell_of in product(range(1, 7), ["STJ", "EDJ"]):
        aggs[f"{i}{spell_of}"] = odds_ratio(i, spell_of)
        aggs[f"{i}{spell_of}_signif"] = is_signif_OR(i, spell_of)
    out = out.select(**aggs)

    out = out.to_numpy().reshape(6, 4)
    overlaps_ = out[:, [0, 2]].T
    pvals_ = out[:, [1, 3]].T

    hoho = (
        pl.DataFrame(overlaps_, schema=DUNCANS_REGIONS_NAMES)
        .with_columns([(pl.col(region)).round(1).cast(pl.String()) for region in DUNCANS_REGIONS_NAMES])
        .with_columns(
            **{
                f"start{region}": pl.when(pl.lit(pl.Series(None, pvals_[:, i]) > 0.95))
                .then(pl.lit(r"$\mathbf{"))
                .otherwise(pl.lit(r"${"))
                for i, region in enumerate(DUNCANS_REGIONS_NAMES)
            }
        )
        .with_columns(
            **{
                region: pl.col(f"start{region}") + pl.col(region) + r"}$"
                for region in DUNCANS_REGIONS_NAMES
            }
        )
        .drop([f"start{region}" for region in DUNCANS_REGIONS_NAMES])
    )
    print(key, hoho)
    hoho.to_pandas().to_latex(
        buf=f"{RESULTS}/OR_{key}.tex",
        escape=False,
        column_format="l",
        multirow=False,
        header=True,
        index_names=False,
    )

### wrs

In [ ]:
wr_names = ["No", "GLBl", "ScTr", "EuBl", "ScBl"]
colors = ["#6C6C6C", "#7E7EF4", "#F2A685", "#8FC386", "green"]

grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")
grams_wr = grams_wr.with_columns(
    **{
        f"winner_{i}": pl.when(pl.col("winner") == i)
        .then(pl.lit(1.0))
        .otherwise(pl.lit(0.0))
        for i in range(5)
    }
)
width = 0.25
fig, axes = plt.subplot_mosaic(
    [["a)", "b)", "c)"], ["a)", "b)", "d)"]],
    figsize=(8, 4),
    constrained_layout=True,
    sharey=True,
    width_ratios=[1, 1, 0.5],
    gridspec_kw=dict(wspace=0.1),
)
for i, (spell_of, ax) in enumerate(zip(["STJ", "EDJ"], list(axes.values()))):
    grams_wr_masked = mask_from_spells_pl(
        spells_list[spell_of].drop("jet ID"), grams_wr, time_before=datetime.timedelta(days=3)
    )
    huh = grams_wr_masked.group_by(["relative_index"]).mean().sort("relative_index")
    alive_spells = (
        grams_wr_masked.group_by("relative_index")
        .agg(pl.col("spell").n_unique())
        .sort("relative_index")["spell"]
        .to_numpy()
    )
    x = huh["relative_index"].to_numpy() / 4
    filter_ = alive_spells > 3
    x = x[filter_]
    bottom = np.zeros(len(x))
    for j in [*np.arange(1, 5), 0]:
        height = huh[f"winner_{j}"].to_numpy()[filter_]
        ax.bar(
            x,
            height,
            bottom=bottom,
            facecolor=colors[j],
            width=width,
            label=wr_names[j],
        )
        bottom = bottom + height
    ax.set_xlabel("Relative time around onset [day]")
    ax.set_title(
        f"{ascii_lowercase[i]}) {alive_spells.max():n} episodes of the {spell_of[:3]}"
    )
    ax.set_xlim(x[0] - width / 2, x[-1] + width / 2)
    ybounds = [1, 1.05]
    im = ax.pcolormesh(
        x,
        ybounds,
        alive_spells[filter_][None, 1:],
        zorder=2,
        cmap=colormaps.greys,
        alpha=0.7,
        vmin=0,
    )
    ax.axhline(1, color="black")
    ax.vlines(0, 0, 1, color="black", ls="dotted", lw=1.2)
h, l = axes["b)"].get_legend_handles_labels()
axes["c)"].set_axis_off()
axes["c)"].legend(h, l, fontsize=12, ncol=1, loc="upper left", title="WR names")
axes["a)"].set_ylabel("WR frequency")
ax = axes["d)"]
monthly_means = (
    grams_wr.filter(pl.col("time").dt.month() > 5)
    .group_by(pl.col("time").dt.month())
    .agg(*[pl.col(f"winner_{i}").mean() for i in range(5)])
    .sort("time")
)
x = np.array([6, 7, 8, 9])
bottom = np.zeros(len(x))
for j in [*np.arange(1, 5), 0]:
    height = monthly_means[f"winner_{j}"].to_numpy()
    ax.bar(x, height, bottom=bottom, facecolor=colors[j], width=0.9, label=wr_names[j])
    bottom = bottom + height
ax.set_xticks(x, labels="JJAS")
ax.set_title("c) Monthly means")
fig.savefig(f"{FIGURES}/Persistence/wrs_bars.pdf")

# props

In [ ]:
from string import ascii_lowercase
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_theta",
    "mean_lev",
    "mean_s",
    "width",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessDC16",
    "wavinessFV15",
    "mean_lat_var",
    "mean_s_var",
    "is_polar",
    "int",
    "int_over_europe",
]

for jet, spell in spells_list.items():
    fig = plot_relative_time(props_catd_summer, spell.drop("jet ID"), data_vars, 1, 4, 4, row_height=2.3, col_width=3.1)
    fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase
data_vars = [
    "t2m-warm",
    "t2m-cold",
    "tp-warm",
    "tp-warm_entrance",
    "tp-warm_far_entrance",
    "tp-cold",
]

for jet, spell in spells_list.items():
    fig = plot_relative_time(props_with_extras_summer, spell.drop("jet ID"), data_vars, 1, 3, 2, row_height=2.3, col_width=3.1, plume=False)
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase
data_vars = [
    "APVO-warm",
    "APVO-cold",
    "CPVO-warm",
    "CPVO-cold",
]

for jet, spell in spells_list.items():
    fig = plot_relative_time(props_with_extras_summer, spell.drop("jet ID"), data_vars, 1, 2, 2, row_height=2.3, col_width=3.1)
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase
data_vars = [
    "EKE250-warm",
    "EKE250-cold",
    "hor-warm",
    "hor-cold",
]

for jet, spell in spells_list.items():
    fig = plot_relative_time(props_with_extras_summer, spell.drop("jet ID"), data_vars, 1, 2, 2, row_height=2.3, col_width=3.1)
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase
data_vars = [
    "PV330-warm",
    "PV330-cold",
    "PV350-warm",
    "PV350-cold",
]

for jet, spell in spells_list.items():
    fig = plot_relative_time(props_with_extras_summer, spell, data_vars, 1, 2, 2, row_height=2.3, col_width=3.1)
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase
data_vars = [
    "PV330:grad-core",
    "PV350:grad-core",
    "theta:grad-core",
    "mean_lat",
    "mean_s",
    "wavinessDC16",
]

for jet, spell in spells_list.items():
    fig = plot_relative_time(props_with_extras_summer, spell, data_vars, 1, 3, 2, row_height=2.5, col_width=3.1)
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

### reduced

In [ ]:
data_vars = [
    "mean_lat",
    "mean_s",
    "wavinessR16",
    "mean_s_var",
]
bigspells = pl.concat(list(spells_list.values()))
fig = plot_relative_time(phat_props_summer, bigspells, data_vars, 2, 2, 2, row_height=2.3, col_width=3.1)
fig.savefig(f"{FIGURES}/Persistence/both_props_30spells.pdf")

# realspace from data

In [ ]:
import polars_st as st
def centroid_jets(jets: pl.DataFrame):
    orig_jets = jets.clone()
    jets = jets.group_by("time", "jet").agg(points=pl.concat_arr(pl.col("lon"), pl.col("lat")))
    points = st.linestring("points").st.set_srid(4326).st.to_srid(3857)
    points_right = st.linestring("points_right").st.set_srid(4326).st.to_srid(3857)
    frechet_expr = points.st.frechet_distance(points_right)
    cross = jets.join(jets, how="cross").select("time", "jet", "time_right", "jet_right", frechet_expr)
    center_jet = cross.group_by("time", "jet").agg(pl.col("points").sum()).sort("points").head(1).join(orig_jets, on="time")
    return center_jet

centroids_path = basepath.joinpath("centroids_during_spells.parquet")
if centroids_path.is_file():
    centroids = pl.read_parquet(centroids_path)
else:
    centroids = []
    for jet, spell_of in product(["STJ", "EDJ"], ["STJ", "EDJ"]):
        jets_ = spells_list[spell_of].select("time").join(phat_jets, on="time").filter(pl.col("jet") == jet).with_columns(spell_of=pl.lit(spell_of))
        centroids.append(centroid_jets(jets_))
    centroids = pl.concat(centroids)
    centroids.write_parquet(centroids_path)

## NATL

In [ ]:
to_plot = xr.open_dataset(basepath.joinpath("NAtl.nc"))
pvals = xr.open_dataset(basepath.joinpath("NAtl_pvals.nc"))
fdr = xr.open_dataset(basepath.joinpath("NAtl_fdr.nc"))

cbar_kwargs = {"shrink": 1.0, "fraction": 0.11, "pad": 0.03}
plot_kwargs_1 = {
    "cmap": colormaps.cmp_b2r,
    "levels": [-1.2, -0.8, -0.4, 0.4, 0.8, 1.2],
    "transparify": 1,
    "cbar_kwargs": cbar_kwargs,
}
plot_kwargs_2 = {
    "cmap": colormaps.brbg,
    "levels": np.linspace(-2, 2, 9).tolist(),
    "transparify": 0,
    "cbar_kwargs": cbar_kwargs,
}
stippling_kwargs = {
    "FDR": True,
    "invert": False,
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xxx",
}
nrow, ncol = 1, 2
days_around = 3
n = nrow * ncol
cmap = colormaps.pastel
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
bigfig = plt.figure(figsize=(6.8, 6), constrained_layout=True)
jets = ["STJ", "EDJ"]
subfigs = bigfig.subfigures(2, 1)
for varname, plot_kwargs, fig in zip(
    ["t2m", "tp"], [plot_kwargs_1, plot_kwargs_2], subfigs
):
    clu = Clusterplot(nrow, ncol, get_region(to_plot), fig=fig)
    factor = FACTORS_UNITS.get(varname, 1)
    titles = []
    for j, jet in enumerate(spells_list):
        letter = ascii_lowercase[
            j % len(ascii_lowercase) + len(clu.axes) * int(varname == "tp")
        ]
        n_spells = spells_list[jet]["spell"].n_unique()
        titles.append(f"{letter}) {jet[:3]}, {n_spells} episodes")
        centroids_ = centroids.filter(pl.col("spell_of") == jet)
        for (_, jetn), jet in centroids_.group_by("time", "jet"):
            if jetn == "EDJ":
                color = COLORS[1]
            else:
                color = COLORS[2]
            clu.axes[j].plot(jet["lon"] - clu.central_longitude, jet["lat"], color=color, lw=3)
    clu.add_contourf([to_plot[f"{varname}_{jet}"] * factor for jet in jets], titles=titles, **plot_kwargs)
    clu.add_stippling(xr.concat([fdr[f"{varname}_{jet}"] for jet in jets], "jet"), **stippling_kwargs)
    # clu.add_contour(
    #     data_contour, levels=[-60, -20, 20, 60], linewidths=2.0, clabels=False
    # )
    fig.suptitle(PRETTIER_VARNAME.get(varname, varname))
bigfig.savefig(f"{FIGURES}/Persistence/t2m_and_tp_realspace_both_30spells_NAtl.pdf")

In [ ]:
from jetutils.stats import fdr_correction
to_plot = xr.open_dataset(basepath.joinpath("NAtl.nc")).sel(lon=slice(-10, 40), lat=slice(30, 70))
pvals = xr.open_dataset(basepath.joinpath("NAtl_pvals.nc")).sel(lon=slice(-10, 40), lat=slice(30, 70))
fdr = xr.Dataset({var: pvals[var].copy(data=fdr_correction(pvals[var].values, 0.05, two_sided=True)) for var in pvals.data_vars})

cbar_kwargs = {"shrink": 1.0, "fraction": 0.11, "pad": 0.03}
plot_kwargs_1 = {
    "cmap": colormaps.cmp_b2r,
    "levels": [-1.2, -0.8, -0.4, 0.4, 0.8, 1.2],
    "transparify": 1,
    "cbar_kwargs": cbar_kwargs,
}
plot_kwargs_2 = {
    "cmap": colormaps.brbg,
    "levels": np.linspace(-2, 2, 9).tolist(),
    "transparify": 0,
    "cbar_kwargs": cbar_kwargs,
}
stippling_kwargs = {
    "FDR": True,
    "invert": False,
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xxx",
}
nrow, ncol = 1, 2
days_around = 3
n = nrow * ncol
cmap = colormaps.pastel
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
bigfig = plt.figure(figsize=(6.8, 6), constrained_layout=True)
jets = ["STJ", "EDJ"]
subfigs = bigfig.subfigures(2, 1)
for varname, plot_kwargs, fig in zip(
    ["t2m", "tp"], [plot_kwargs_1, plot_kwargs_2], subfigs
):
    clu = Clusterplot(nrow, ncol, get_region(to_plot), fig=fig)
    factor = FACTORS_UNITS.get(varname, 1)
    titles = []
    for j, jet in enumerate(spells_list):
        letter = ascii_lowercase[
            j % len(ascii_lowercase) + len(clu.axes) * int(varname == "tp")
        ]
        n_spells = daily_spells_list[jet]["spell"].n_unique()
        titles.append(f"{letter}) {jet[:3]}, {n_spells} episodes")
        centroids_ = centroids.filter(pl.col("spell_of") == jet)
        for (_, jetn), jet in centroids_.group_by("time", "jet"):
            if jetn == "EDJ":
                color = COLORS[1]
            else:
                color = COLORS[2]
            clu.axes[j].plot(jet["lon"] - clu.central_longitude, jet["lat"], color=color, lw=3)
    clu.add_contourf([to_plot[f"{varname}_{jet}"] * factor for jet in jets], titles=titles, **plot_kwargs)
    clu.add_stippling(xr.concat([fdr[f"{varname}_{jet}"] for jet in jets], "jet"), **stippling_kwargs)
    # clu.add_contour(
    #     data_contour, levels=[-60, -20, 20, 60], linewidths=2.0, clabels=False
    # )
    fig.suptitle(PRETTIER_VARNAME.get(varname, varname))
bigfig.savefig(f"{FIGURES}/Persistence/t2m_and_tp_realspace_both_30spells_Europe.pdf")

## in europe

center jets for STJ, EDJ for spells of STJ, EDJ. 
versus
jets on wind centroids

plot europe AND NAtl composties with jets (either version), and maybe z500p

at home later today:
re-do case study plots locally

In [ ]:
figure_data = f"{FIGURES}/Persistence/figure_data/realspace_fig06"
data_contour = xr.open_dataarray(f"{figure_data}/contour.nc")
data_signif = xr.open_dataset(f"{figure_data}/signifs.nc")
data_wind = xr.open_dataset(f"{figure_data}/wind.nc")
data_contourf = xr.open_dataset(f"{figure_data}/contourf.nc")

find_jets_kwargs = dict(
    n_coarsen=1,
    base_s_thresh=24,
    alignment_thresh=0.4,
    int_thresh_factor=0.1,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets_on_mean = find_all_jets(data_wind, **find_jets_kwargs)
jets_on_mean = jets_on_mean.with_columns(spell_of=pl.when(pl.col("relative_index") == 1).then(pl.lit("EDJ")).otherwise(pl.lit("STJ")))
cat_vitaif = pl.col("theta").mean().over("spell_of", "jet ID") > 340
cat_vitaif = (
    pl.when(cat_vitaif).then(pl.lit("STJ")).otherwise(pl.lit("EDJ"))
)
jets_on_mean = jets_on_mean.with_columns(jet=cat_vitaif)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 7))
for spell_of, ax in zip(["STJ", "EDJ"], axes):
    for i, jet in enumerate(["STJ", "EDJ"]):
        jc = centroids.filter(pl.col("jet") == jet, pl.col("spell_of") == spell_of)
        ax.scatter(jc["lon"], jc["lat"], color=COLORS[2 - i])
        
        jom = jets_on_mean.filter(pl.col("spell_of") == spell_of, pl.col("jet") == jet)
        ax.scatter(jom["lon"], jom["lat"], color=COLORS_EXT[8 - 3 * i])